# Gold Layer

Prepare data for analysis and machine learning algorithms by aggregating and feature engineering.

In [0]:
%run ./0_config

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import IntegerType

SILVER_TABLE = "workspace.default.energy_clean"
GOLD_HOURLY_TABLE = "workspace.default.demand_hourly"
GOLD_FORECAST_TABLE = "workspace.default.demand_forecast"

def read_silver():
    df = spark.read.table(SILVER_TABLE)
    return df

def build_hourly_features(df):
    # format data by hour (and region if more regions are added)
    df = df.withColumn("hour_ts", F.date_trunc("hour", "settlement_ts"))
    df = df.groupBy("hour_ts", "region").agg(
        F.mean("total_demand_mw").alias("avg_demand_mw"),
        F.max("total_demand_mw").alias("max_demand_mw"),
        F.mean("price_rrp").alias("avg_price_rrp")
    )

    # add features for ml model
    df = df.select(
        "*",
        F.to_date("hour_ts").alias("date"),
        F.hour("hour_ts").cast(IntegerType()).alias("hour"),
        F.dayofweek("hour_ts").cast(IntegerType()).alias("day_of_week"),
        F.month("hour_ts").cast(IntegerType()).alias("month"),
        F.quarter("hour_ts").cast(IntegerType()).alias("quarter"),
        F.year("hour_ts").cast(IntegerType()).alias("year"),
        (F.dayofweek("hour_ts").isin([1, 7])).cast(IntegerType()).alias("is_weekend"),
        F.when(F.month("hour_ts").isin([12, 1, 2]), "summer")
         .when(F.month("hour_ts").isin([3, 4, 5]), "autumn")
         .when(F.month("hour_ts").isin([6, 7, 8]), "winter")
         .otherwise("spring").alias("season")
    )

    df = df.orderBy("hour_ts")  # sort for time series analysis
    return df

def write_gold_features(df):
    (df.write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .saveAsTable(GOLD_HOURLY_TABLE))
    print("[GOLD] hourly features written")

def write_gold_forecast_placeholder(df):
    forecast_df = (df
        .select("hour_ts", "region", F.col("avg_demand_mw").alias("actual_demand_mw"))
        .limit(0)
        .withColumn("predicted_demand_mw", F.lit(None).cast("double"))
        .withColumn("model_version", F.lit("v0"))
        .withColumn("created_at", F.current_timestamp())
    )
    (forecast_df.write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .saveAsTable(GOLD_FORECAST_TABLE))
    print("[GOLD] forecast placeholder written")

def main(df = None):
    if df is None:
        df = build_hourly_features(read_silver())
    
    write_gold_features(df)
    write_gold_forecast_placeholder(df)

## Aggregation

Run the preview block before `main` to execute the final layer.

### Running

Create the gold layer dataframe and check the schema before writing it to volumes.

In [0]:
df_silver = read_silver()
df_gold = build_hourly_features(df_silver)
df_gold.printSchema()
df_gold.show(5)

In [0]:
main(df_gold)